# Student Performance Analysis and Prediction Using Data Mining Techniques

This notebook follows a BCA-level Data Mining case study using the UCI Student Performance (Mathematics) dataset.

The workflow is:
1. Load the dataset and verify quality
2. Create the target variable `Performance`
3. Perform exploratory data analysis
4. Build and compare classification models
5. Apply clustering with validation
6. Save results for the final report and presentation


In [ ]:
import os
os.environ['MPLCONFIGDIR'] = os.path.join(os.getcwd(), '.matplotlib_cache')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram

import os

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')

print('Libraries imported successfully.')
print('Random state set to:', RANDOM_STATE)


In [ ]:
# Load the dataset
file_path = 'student-mat.csv'

if not os.path.exists(file_path):
    raise FileNotFoundError(
        "The dataset file 'student-mat.csv' was not found in the current folder. "
        "Please download it from the UCI repository and save it in the same folder as this notebook."
    )

df = pd.read_csv(file_path, sep=';')

print('Dataset shape:', df.shape)
print('Columns:', list(df.columns))
print('\nData types:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nG3 descriptive statistics:\n', df['G3'].describe())


In [ ]:
# Create the target variable 'Performance'
df['Performance'] = pd.cut(
    df['G3'],
    bins=[-1, 9, 14, 20],
    labels=['Low', 'Medium', 'High'],
    include_lowest=True
)

performance_counts = df['Performance'].value_counts().sort_index()
print('Performance distribution:\n', performance_counts)

plt.figure(figsize=(8, 5), dpi=300)
sns.countplot(data=df, x='Performance', order=['Low', 'Medium', 'High'], palette='Set2')
plt.title('Student Performance Distribution')
plt.xlabel('Performance Category')
plt.ylabel('Number of Students')
plt.tight_layout()
plt.show()


In [ ]:
# EDA 1: G3 distribution
plt.figure(figsize=(8, 5), dpi=300)
sns.histplot(df['G3'], bins=20, kde=True, color='steelblue')
plt.title('Distribution of Final Mathematics Grade (G3)')
plt.xlabel('Final Grade (G3)')
plt.ylabel('Number of Students')
plt.tight_layout()
plt.show()

# EDA 2: Absences vs G3
plt.figure(figsize=(8, 5), dpi=300)
sns.scatterplot(data=df, x='absences', y='G3', alpha=0.7, color='darkorange')
plt.title('Absences vs Final Mathematics Grade (G3)')
plt.xlabel('Number of Absences')
plt.ylabel('Final Grade (G3)')
plt.tight_layout()
plt.show()

# EDA 3: Study time vs G3
plt.figure(figsize=(8, 5), dpi=300)
sns.boxplot(data=df, x='studytime', y='G3', palette='Set2')
plt.title('Study Time vs Final Mathematics Grade (G3)')
plt.xlabel('Study Time Category')
plt.ylabel('Final Grade (G3)')
plt.tight_layout()
plt.show()

# EDA 4: Correlation heatmap
numeric_cols = df.select_dtypes(include=['number']).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 10), dpi=300)
sns.heatmap(corr, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()


In [ ]:
# Define feature matrix and target
# G3 is excluded because it directly defines the target.
# G1 and G2 are also excluded in the main experiment to avoid direct grade leakage.
X = df.drop(columns=['G3', 'Performance', 'G1', 'G2'])
y = df['Performance']

print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nFirst 5 rows of X:')
print(X.head())
print('\nTarget distribution:')
print(y.value_counts())


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))
print('y_train distribution:\n', y_train.value_counts())
print('\ny_test distribution:\n', y_test.value_counts())

# Keep track of categorical and numerical columns for preprocessing
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['number']).columns.tolist()

print('\nCategorical columns:', len(categorical_cols))
print(categorical_cols)
print('\nNumerical columns:', len(numerical_cols))
print(numerical_cols)


In [ ]:
# Preprocessing pipeline setup
# This uses a ColumnTransformer so categorical and numerical features are handled separately.
preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('numerical', StandardScaler(), numerical_cols)
    ]
)

print('Preprocessor created successfully.')
print(preprocessor)


In [ ]:
from sklearn.base import clone


def evaluate_model(model_name, model, X_train, y_train, X_test, y_test, class_labels=None):
    """Fit a model, produce predictions, and return evaluation metrics."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    print(f'\n=== {model_name} ===')
    print('Accuracy:', acc)
    print('Weighted Precision:', prec)
    print('Weighted Recall:', rec)
    print('Weighted F1-score:', f1)
    print('\nClassification report:\n')
    print(classification_report(y_test, y_pred, zero_division=0, labels=class_labels))

    cm = confusion_matrix(y_test, y_pred, labels=class_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
    disp.plot(cmap='Blues')
    plt.title(f'{model_name} Confusion Matrix')
    plt.tight_layout()
    plt.show()

    return {
        'model': model_name,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'y_pred': y_pred,
        'cm': cm
    }


class_labels = ['Low', 'Medium', 'High']
print('Helper functions ready.')


In [ ]:
# Decision Tree (entropy-based ID3-style)
decision_tree = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=RANDOM_STATE))
])

dt_result = evaluate_model(
    'Decision Tree',
    decision_tree,
    X_train,
    y_train,
    X_test,
    y_test,
    class_labels=class_labels
)

y_pred_dt = dt_result['y_pred']
print('\nStored y_pred_dt for later use.')


In [ ]:
# Gaussian Naive Bayes
naive_bayes = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GaussianNB())
])

nb_result = evaluate_model(
    'Naive Bayes',
    naive_bayes,
    X_train,
    y_train,
    X_test,
    y_test,
    class_labels=class_labels
)

y_pred_nb = nb_result['y_pred']
print('\nStored y_pred_nb for later use.')


In [ ]:
# K-Nearest Neighbors (k=7)
knn = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=7))
])

knn_result = evaluate_model(
    'KNN',
    knn,
    X_train,
    y_train,
    X_test,
    y_test,
    class_labels=class_labels
)

y_pred_knn = knn_result['y_pred']
print('\nStored y_pred_knn for later use.')


In [ ]:
# Support Vector Machine (RBF kernel)
svm_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SVC(kernel='rbf', random_state=RANDOM_STATE))
])

svm_result = evaluate_model(
    'SVM',
    svm_model,
    X_train,
    y_train,
    X_test,
    y_test,
    class_labels=class_labels
)

y_pred_svm = svm_result['y_pred']
print('\nStored y_pred_svm for later use.')


In [ ]:
# Compare the classifier metrics
model_metrics = pd.DataFrame([
    dt_result,
    nb_result,
    knn_result,
    svm_result
]).drop(columns=['y_pred', 'cm'])

# Reorder models to fit report expectations
model_metrics = model_metrics[['model', 'accuracy', 'precision', 'recall', 'f1']]
print(model_metrics)

# Grouped bar chart for model comparison
plt.figure(figsize=(10, 6), dpi=300)
metrics_for_plot = model_metrics.set_index('model')
metrics_for_plot.plot(kind='bar', figsize=(10, 6), width=0.8)
plt.title('Classification Model Comparison')
plt.ylabel('Score')
plt.xlabel('Model')
plt.xticks(rotation=0)
plt.ylim(0, 1.0)
plt.legend(title='Metric')
plt.tight_layout()
plt.show()


In [ ]:
# Save classifier metrics for the final report
os.makedirs('results', exist_ok=True)
model_metrics.to_csv('results/classifier_metrics.csv', index=False)
print('Saved classifier metrics to results/classifier_metrics.csv')


In [ ]:
# Clustering dataset selection
cluster_features = ['age', 'Medu', 'Fedu', 'studytime', 'failures', 'absences', 'famrel', 'freetime', 'goout', 'health']
cluster_data = df[cluster_features].copy()

print('Cluster data shape:', cluster_data.shape)
print(cluster_data.head())

# Standardize the clustering variables
cluster_scaler = StandardScaler()
cluster_scaled = cluster_scaler.fit_transform(cluster_data)

print('\nScaled cluster data shape:', cluster_scaled.shape)
print('\nSample of scaled cluster data:')
print(cluster_scaled[:5])


In [ ]:
# K-Means silhouette analysis for K = 2 to 6
silhouette_scores = []
for k in range(2, 7):
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = kmeans.fit_predict(cluster_scaled)
    score = silhouette_score(cluster_scaled, labels)
    silhouette_scores.append((k, score))
    print(f'K = {k}, Silhouette Score = {score:.4f}')

k_values = [k for k, _ in silhouette_scores]
scores = [score for _, score in silhouette_scores]

plt.figure(figsize=(8, 5), dpi=300)
plt.plot(k_values, scores, marker='o', color='royalblue', linewidth=2)
plt.title('Silhouette Score vs Number of Clusters (K)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.xticks(k_values)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

best_k = k_values[np.argmax(scores)]
print('\nSelected best K based on silhouette score:', best_k)


In [ ]:
# Final K-Means model using the best K
final_kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = final_kmeans.fit_predict(cluster_scaled)

cluster_sizes = pd.Series(kmeans_labels).value_counts().sort_index()
print('Cluster sizes:\n', cluster_sizes)

# Inverse-transform centroids to original scale for interpretation
centroids_original = cluster_scaler.inverse_transform(final_kmeans.cluster_centers_)
centroids_df = pd.DataFrame(centroids_original, columns=cluster_features)
print('\nCluster centroids (original scale):\n')
print(centroids_df.round(2))


In [ ]:
# PCA for K-Means visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(cluster_scaled)

plt.figure(figsize=(8, 6), dpi=300)
for cluster_id in range(best_k):
    mask = kmeans_labels == cluster_id
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Cluster {cluster_id}', alpha=0.7)

centroids_pca = pca.transform(final_kmeans.cluster_centers_)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='black', marker='X', s=80, label='Centroid')

plt.title('PCA Visualization of K-Means Clusters')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Agglomerative Hierarchical Clustering
# Using Ward linkage as a common hierarchical method
ward_linkage = linkage(cluster_scaled, method='ward')

plt.figure(figsize=(12, 8), dpi=300)
dendrogram(ward_linkage, truncate_mode='level', p=8)
plt.title('Dendrogram for Agglomerative Hierarchical Clustering')
plt.xlabel('Sample Index')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

agg_model = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
agg_labels = agg_model.fit_predict(cluster_scaled)
agg_sil = silhouette_score(cluster_scaled, agg_labels)
print(f'Agglomerative silhouette score for K={best_k}: {agg_sil:.4f}')
print('\nAgglomerative cluster sizes:\n', pd.Series(agg_labels).value_counts().sort_index())


In [ ]:
# External validation: compare cluster labels to known performance labels
# Performance is not used to build the clusters; it is used only after clustering.
performance_numeric = df['Performance'].map({'Low': 0, 'Medium': 1, 'High': 2})

ari = adjusted_rand_score(performance_numeric, kmeans_labels)
nmi = normalized_mutual_info_score(performance_numeric, kmeans_labels)

print('Adjusted Rand Index (ARI):', round(ari, 4))
print('Normalized Mutual Information (NMI):', round(nmi, 4))

# Optional: compare agglomerative labels as well
agg_ari = adjusted_rand_score(performance_numeric, agg_labels)
agg_nmi = normalized_mutual_info_score(performance_numeric, agg_labels)
print('Agglomerative ARI:', round(agg_ari, 4))
print('Agglomerative NMI:', round(agg_nmi, 4))


In [1]:
# Save clustering results for the final report
os.makedirs('results', exist_ok=True)

cluster_summary = {
    'best_k': int(best_k),
    'kmeans_silhouette': float(silhouette_score(cluster_scaled, kmeans_labels)),
    'agglomerative_silhouette': float(agg_sil),
    'ari': float(ari),
    'nmi': float(nmi),
    'cluster_sizes': cluster_sizes.to_dict(),
    'performance_distribution': df['Performance'].value_counts().to_dict(),
}

pd.Series(cluster_summary).to_json('results/clustering_summary.json')

# Save labels for later analysis
cluster_df = pd.DataFrame({
    'student_index': df.index,
    'kmeans_cluster': kmeans_labels,
    'agglomerative_cluster': agg_labels,
    'Performance': df['Performance']
})
cluster_df.to_csv('results/cluster_labels.csv', index=False)

print('Clustering results saved to results/clustering_summary.json and results/cluster_labels.csv')


NameError: name 'os' is not defined